# Notebook 02: Preprocessing and ML Data Preparation

This notebook loads `merged_banner_moodle_with_target.pkl`, removes identifiers and leakage, selects early prediction features, converts data types, creates a stratified train and test split, encodes categories, scales numeric features, and saves the prepared data.


## 1. Import libraries


In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the merged dataset


In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INPUT_FILE = PROJECT_DIR / "data" / "processed" / "merged_banner_moodle_with_target.pkl"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed" 

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

data = pd.read_pickle(INPUT_FILE)

print("Loaded:", INPUT_FILE)
print("Shape:", data.shape)


Loaded: c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\data\processed\merged_banner_moodle_with_target.pkl
Shape: (3760, 40)


## 3. Confirm the target


In [3]:
TARGET = "academic_risk_label"

if TARGET not in data.columns:
    raise KeyError(f"Target column not found: {TARGET}")

data = data.dropna(subset=[TARGET]).reset_index(drop=True)
data[TARGET] = data[TARGET].astype(int)

display(
    data[TARGET]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET)
    .to_frame("count")
)


,count
academic_risk_label,
0,2827
1,933


In [4]:
data = data.drop(columns=["merge_student_key_banner", "merge_student_key_moodle"])

In [5]:
# Count fields containing 0, 1, 2, 3, 4, and 5+
CAPPED_COUNT_COLUMNS = [
    "repeated_course_count_current",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",
]

for column in CAPPED_COUNT_COLUMNS:

    cleaned_values = (
        data[column]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("5+", "5", regex=False)
    )

    data[column] = pd.to_numeric(
        cleaned_values,
        errors="coerce",
    )

print("Capped count fields converted successfully.")

display(
    data[CAPPED_COUNT_COLUMNS]
    .describe()
    .T
)

Capped count fields converted successfully.


,count,mean,std,min,25%,50%,75%,max
repeated_course_count_current,3760.0,0.38883,0.819348,0.0,0.0,0.0,0.0,5.0
previous_failed_course_count,3760.0,0.800266,1.399369,0.0,0.0,0.0,1.0,5.0
previous_withdrawn_course_count,3760.0,0.074468,0.380821,0.0,0.0,0.0,0.0,5.0
previous_repeated_course_count,3760.0,0.288564,0.918885,0.0,0.0,0.0,0.0,5.0


In [6]:
#Convert active_days to numeric
active_days_clean = (
    data["active_days"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "3 or fewer": "3",
    })
)

data["active_days_numeric"] = pd.to_numeric(
    active_days_clean,
    errors="raise",
)

print(
    data["active_days_numeric"]
    .describe()
)

count      3760.0
mean     9.897074
std      6.439461
min           3.0
25%           3.0
50%           9.0
75%          14.0
max          30.0
Name: active_days_numeric, dtype: Float64


In [7]:
# =========================================================
# CREATE DATA DICTIONARY FOR ALL COLUMNS
# =========================================================

IDENTIFIER_COLUMNS = {
    "student_key"
}

MAX_CATEGORIES_TO_SHOW = 20
TOP_VALUES_TO_SHOW = 5

dictionary_rows = []

for column in data.columns:

    series = data[column]

    missing_count = int(series.isna().sum())
    missing_percentage = round(
        series.isna().mean() * 100,
        2,
    )

    unique_count = int(
        series.nunique(dropna=True)
    )

    # -----------------------------------------------------
    # Hide identifiers
    # -----------------------------------------------------
    if column in IDENTIFIER_COLUMNS:

        column_type = "Identifier"
        value_summary = "Values hidden for privacy"
        common_values = "Not displayed"

    # -----------------------------------------------------
    # Numeric columns
    # -----------------------------------------------------
    elif pd.api.types.is_numeric_dtype(series):

        numeric_values = pd.to_numeric(
            series,
            errors="coerce",
        )

        column_type = "Numeric"

        if numeric_values.notna().any():

            value_summary = (
                f"Min: {numeric_values.min():.3f}, "
                f"Max: {numeric_values.max():.3f}, "
                f"Mean: {numeric_values.mean():.3f}, "
                f"Median: {numeric_values.median():.3f}"
            )

        else:
            value_summary = "No valid numeric values"

        common_values = (
            series
            .value_counts(dropna=False)
            .head(TOP_VALUES_TO_SHOW)
            .to_dict()
        )

    # -----------------------------------------------------
    # Non numeric columns
    # -----------------------------------------------------
    else:

        cleaned_series = (
            series
            .astype("string")
            .str.strip()
        )

        # Check whether the column contains numeric values
        numeric_test = (
            cleaned_series
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric_test = pd.to_numeric(
            numeric_test,
            errors="coerce",
        )

        non_missing_count = int(
            cleaned_series.notna().sum()
        )

        numeric_success_rate = (
            numeric_test.notna().sum()
            / non_missing_count
            if non_missing_count > 0
            else 0
        )

        # Numeric values currently stored as text
        if numeric_success_rate >= 0.90:

            column_type = "Numeric stored as text"

            value_summary = (
                f"Min: {numeric_test.min():.3f}, "
                f"Max: {numeric_test.max():.3f}, "
                f"Mean: {numeric_test.mean():.3f}, "
                f"Median: {numeric_test.median():.3f}"
            )

            common_values = (
                cleaned_series
                .value_counts(dropna=False)
                .head(TOP_VALUES_TO_SHOW)
                .to_dict()
            )

        # Normal categorical column
        elif unique_count <= MAX_CATEGORIES_TO_SHOW:

            column_type = "Categorical"

            categories = (
                cleaned_series
                .dropna()
                .unique()
                .tolist()
            )

            value_summary = " | ".join(
                map(str, categories)
            )

            common_values = (
                cleaned_series
                .value_counts(dropna=False)
                .head(TOP_VALUES_TO_SHOW)
                .to_dict()
            )

        # High cardinality text column
        else:

            column_type = "High cardinality categorical or text"

            sample_values = (
                cleaned_series
                .dropna()
                .unique()[:TOP_VALUES_TO_SHOW]
                .tolist()
            )

            value_summary = (
                "Sample values: "
                + " | ".join(
                    map(str, sample_values)
                )
            )

            common_values = (
                cleaned_series
                .value_counts(dropna=False)
                .head(TOP_VALUES_TO_SHOW)
                .to_dict()
            )

    dictionary_rows.append({
        "column": column,
        "original_dtype": str(series.dtype),
        "interpreted_type": column_type,
        "row_count": len(data),
        "missing_count": missing_count,
        "missing_percentage": missing_percentage,
        "unique_count": unique_count,
        "range_or_categories": value_summary,
        "top_values_and_counts": str(common_values),
    })


data_dictionary = pd.DataFrame(
    dictionary_rows
)

display(data_dictionary)

,column,original_dtype,interpreted_type,row_count,missing_count,missing_percentage,unique_count,range_or_categories,top_values_and_counts
0,student_key,string,Identifier,3760,0,0.00,3760,Values hidden for privacy,Not displayed
1,semester_code_banner,int64,Numeric,3760,0,0.00,1,"Min: 202502.000, Max: 202502.000, Mean: 202502...",{202502: 3760}
2,programme_or_school,str,Categorical,3760,0,0.00,6,Bachelor of Business | Bachelor of Engineering...,"{'Bachelor of Engineering': 1520, 'Bachelor of..."
3,year_level,int64,Numeric,3760,0,0.00,4,"Min: 1.000, Max: 4.000, Mean: 2.559, Median: 3...","{3: 985, 4: 983, 2: 942, 1: 850}"
4,gender,str,Categorical,3760,0,0.00,3,Female | Male | Not Disclosed,"{'Male': 2511, 'Female': 1133, 'Not Disclosed'..."
5,registered_course_count,int64,Numeric,3760,0,0.00,9,"Min: 1.000, Max: 9.000, Mean: 4.187, Median: 4...","{4: 1652, 5: 640, 3: 631, 6: 489, 2: 229}"
6,registered_credits,int64,Numeric,3760,0,0.00,23,"Min: 10.000, Max: 125.000, Mean: 58.618, Media...","{60: 1848, 45: 861, 75: 377, 65: 147, 50: 130}"
7,repeated_course_count_current,Int64,Numeric,3760,0,0.00,6,"Min: 0.000, Max: 5.000, Mean: 0.389, Median: 0...","{np.int64(0): 2833, np.int64(1): 590, np.int64..."
8,previous_term_gpa,str,Categorical,3760,303,8.06,6,High | Good | Low | Near Threshold | Very Low ...,"{'High': 1850, 'Good': 1237, <NA>: 303, 'Near ..."
9,previous_term_gpa_range,str,Categorical,3760,303,8.06,6,3.00 to 4.00 | 2.50 to <3.00 | 1.00 to <2.00 |...,"{'3.00 to 4.00': 1850, '2.50 to <3.00': 1237, ..."


In [8]:
DATA_DICTIONARY_FILE = (
    PROJECT_DIR
    / "results"
    / "tables"
    / "data_dictionary.xlsx"
)

DATA_DICTIONARY_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

data_dictionary.to_excel(
    DATA_DICTIONARY_FILE,
    index=False,
)

print("Data dictionary saved to:")
print(DATA_DICTIONARY_FILE)

Data dictionary saved to:
c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\results\tables\data_dictionary.xlsx


## 4. Define excluded and selected features


In [9]:
#Convert previous term GPA and previous CGPA separately
def normalise_category(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("_", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )


GPA_RANGE_MAP = {
    "0.00 TO <1.00": 0,
    "1.00 TO <2.00": 1,
    "2.00 TO <2.25": 2,
    "2.25 TO <2.50": 3,
    "2.50 TO <3.00": 4,
    "3.00 TO 4.00": 5,
}


data["previous_term_gpa_ordinal"] = (
    normalise_category(
        data["previous_term_gpa_range"]
    )
    .map(GPA_RANGE_MAP)
)

data["previous_cgpa_ordinal"] = (
    normalise_category(
        data["previous_cgpa_range"]
    )
    .map(GPA_RANGE_MAP)
)

In [10]:
#Convert the two Moodle engagement categories
ENGAGEMENT_MAP = {
    "VERY LOW": 0,
    "LOW": 1,
    "MODERATE": 2,
    "GOOD": 3,
    "HIGH": 4,
}


data["learning_material_events_ordinal"] = (
    normalise_category(
        data["learning_material_events"]
    )
    .map(ENGAGEMENT_MAP)
)

data["assessment_interaction_events_ordinal"] = (
    normalise_category(
        data["assessment_interaction_events"]
    )
    .map(ENGAGEMENT_MAP)
)

In [ ]:
TARGET = "academic_risk_label"


# Identifiers must not be used for modelling
IDENTIFIER_COLUMNS = [
    "student_key",
    "merge_student_key_banner",
    "merge_student_key_moodle",
]


# Retained separately for fairness evaluation
FAIRNESS_ONLY_COLUMNS = [
    "gender",
]


# End of semester fields and target construction fields
LEAKAGE_COLUMNS = [
    "end_term_gpa",
    "end_cgpa",
    "end_term_gpa_range",
    "end_cgpa_range",
    "earned_credits_this_semester",
    "credits_earned_ratio",
    "credit_deficit",
    "risk_source",
]


# Administrative fields that are not useful predictors
NON_PREDICTIVE_COLUMNS = [
    "semester_code",
    "semester_code_banner",
    "semester_code_moodle",
]


# Original columns already replaced by converted versions
CONVERTED_SOURCE_COLUMNS = [
    "active_days",
    "previous_term_gpa",
    "previous_term_gpa_range",
    "previous_cgpa",
    "previous_cgpa_range",
    "learning_material_events",
    "assessment_interaction_events",
]


# Final numeric features
NUMERIC_CANDIDATES = [
    "registered_course_count",
    "registered_credits",
    "repeated_course_count_current",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",

    # Converted previous academic performance fields
    "previous_term_gpa_ordinal",
    "previous_cgpa_ordinal",

    # Attendance features
    "attendance_rate",
    "absence_count",

    # Moodle course access features
    "enrolled_course_count",
    "accessed_course_count",
    "course_access_rate",
    "total_course_event_clicks",

    # Converted Moodle activity features
    "active_days_numeric",
    "active_days_rate",
    "zero_activity_days",
    "largest_inactivity_days",
    "learning_material_events_ordinal",
    "assessment_interaction_events_ordinal",
]


# Final categorical features
CATEGORICAL_CANDIDATES = [
    "programme_or_school",
    "year_level",
    "previous_academic_standing",
]


# Keep only columns that exist in the dataset
numeric_features = [
    column
    for column in NUMERIC_CANDIDATES
    if column in data.columns
]

categorical_features = [
    column
    for column in CATEGORICAL_CANDIDATES
    if column in data.columns
]

selected_features = (
    numeric_features
    + categorical_features
)


# Collect all excluded fields that exist
excluded_columns = (
    IDENTIFIER_COLUMNS
    + FAIRNESS_ONLY_COLUMNS
    + LEAKAGE_COLUMNS
    + NON_PREDICTIVE_COLUMNS
    + CONVERTED_SOURCE_COLUMNS
    + [TARGET]
)

existing_excluded_columns = [
    column
    for column in excluded_columns
    if column in data.columns
]


# Check for expected features that are missing
missing_numeric_features = [
    column
    for column in NUMERIC_CANDIDATES
    if column not in data.columns
]

missing_categorical_features = [
    column
    for column in CATEGORICAL_CANDIDATES
    if column not in data.columns
]


print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal selected features:")
print(len(selected_features))

print("\nExcluded columns found:")
print(existing_excluded_columns)

print("\nMissing expected numeric features:")
print(missing_numeric_features)

print("\nMissing expected categorical features:")
print(missing_categorical_features)

Numeric features:
['registered_course_count', 'registered_credits', 'repeated_course_count_current', 'previous_failed_course_count', 'previous_withdrawn_course_count', 'previous_repeated_course_count', 'previous_term_gpa_ordinal', 'previous_cgpa_ordinal', 'attendance_rate', 'absence_count', 'enrolled_course_count', 'accessed_course_count', 'course_access_rate', 'total_course_event_clicks', 'active_days_numeric', 'active_days_rate', 'zero_activity_days', 'largest_inactivity_days', 'learning_material_events_ordinal', 'assessment_interaction_events_ordinal']

Categorical features:
['programme_or_school', 'year_level', 'previous_academic_standing']

Total selected features:
23

Excluded columns found:
['student_key', 'gender', 'end_term_gpa', 'end_cgpa', 'end_term_gpa_range', 'end_cgpa_range', 'earned_credits_this_semester', 'credits_earned_ratio', 'credit_deficit', 'risk_source', 'semester_code_banner', 'semester_code_moodle', 'active_days', 'previous_term_gpa', 'previous_term_gpa_range',

## 5. Convert numeric and categorical data types


## 6. Create X, y, and fairness data


In [13]:
if not selected_features:
    raise ValueError("No selected features were found. Check the actual column names.")

X = data[selected_features].copy()
y = data[TARGET].copy()

fairness_data = (
    data[["gender"]].copy()
    if "gender" in data.columns
    else pd.DataFrame(index=data.index)
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fairness data shape:", fairness_data.shape)


X shape: (3760, 23)
y shape: (3760,)
Fairness data shape: (3760, 1)


## 7. Create stratified training and testing sets


In [14]:
indices = np.arange(len(data))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

fairness_train = fairness_data.iloc[train_indices].copy()
fairness_test = fairness_data.iloc[test_indices].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class percentage:")
display(
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)

print("Testing class percentage:")
display(
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)


Training rows: 3008
Testing rows: 752

Training class percentage:


,percentage
academic_risk_label,
0,75.2
1,24.8


Testing class percentage:


,percentage
academic_risk_label,
0,75.13
1,24.87


## 8. Build the preprocessing pipeline


In [15]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Preprocessing pipeline created.")


Preprocessing pipeline created.


## 9. Fit on training data and transform both sets


In [16]:
X_train_prepared_array = preprocessor.fit_transform(X_train)
X_test_prepared_array = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_prepared = pd.DataFrame(
    X_train_prepared_array,
    columns=feature_names,
    index=X_train.index,
)

X_test_prepared = pd.DataFrame(
    X_test_prepared_array,
    columns=feature_names,
    index=X_test.index,
)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared testing shape:", X_test_prepared.shape)
print("Prepared feature count:", len(feature_names))


Prepared training shape: (3008, 35)
Prepared testing shape: (752, 35)
Prepared feature count: 35


## 10. Validate the prepared data


In [17]:
validation = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(X_train_prepared), len(X_test_prepared)],
    "columns": [X_train_prepared.shape[1], X_test_prepared.shape[1]],
    "missing_values": [
        int(X_train_prepared.isna().sum().sum()),
        int(X_test_prepared.isna().sum().sum()),
    ],
})

display(validation)

assert X_train_prepared.shape[1] == X_test_prepared.shape[1]
assert X_train_prepared.isna().sum().sum() == 0
assert X_test_prepared.isna().sum().sum() == 0

print("Prepared data validation completed successfully.")


,dataset,rows,columns,missing_values
0,train,3008,35,0
1,test,752,35,0


Prepared data validation completed successfully.


In [18]:
print("Original numeric features:", len(numeric_features))
print("Original categorical features:", len(categorical_features))
print("Original total features:", len(selected_features))

print("\nPrepared features:")
for number, feature in enumerate(feature_names, start=1):
    print(number, feature)

Original numeric features: 20
Original categorical features: 3
Original total features: 23

Prepared features:
1 registered_course_count
2 registered_credits
3 repeated_course_count_current
4 previous_failed_course_count
5 previous_withdrawn_course_count
6 previous_repeated_course_count
7 previous_term_gpa_ordinal
8 previous_cgpa_ordinal
9 attendance_rate
10 absence_count
11 enrolled_course_count
12 accessed_course_count
13 course_access_rate
14 total_course_event_clicks
15 active_days_numeric
16 active_days_rate
17 zero_activity_days
18 largest_inactivity_days
19 learning_material_events_ordinal
20 assessment_interaction_events_ordinal
21 programme_or_school_Bachelor of Business
22 programme_or_school_Bachelor of Engineering
23 programme_or_school_Bachelor of Film and Animation
24 programme_or_school_Bachelor of Information and Communications Technology
25 programme_or_school_Bachelor of International Logistics Management
26 programme_or_school_Bachelor of Web Media
27 year_level_1
28

## 11. Save outputs for Notebook 03


In [19]:
X_train.to_pickle(OUTPUT_DIR / "X_train_raw.pkl")
X_test.to_pickle(OUTPUT_DIR / "X_test_raw.pkl")

X_train_prepared.to_pickle(OUTPUT_DIR / "X_train_prepared.pkl")
X_test_prepared.to_pickle(OUTPUT_DIR / "X_test_prepared.pkl")

y_train.to_pickle(OUTPUT_DIR / "y_train.pkl")
y_test.to_pickle(OUTPUT_DIR / "y_test.pkl")

fairness_train.to_pickle(OUTPUT_DIR / "fairness_train.pkl")
fairness_test.to_pickle(OUTPUT_DIR / "fairness_test.pkl")

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

pd.DataFrame({
    "feature_name": feature_names
}).to_csv(
    OUTPUT_DIR / "prepared_feature_names.csv",
    index=False,
)

summary = {
    "input_rows": int(len(data)),
    "raw_feature_count": int(len(selected_features)),
    "numeric_feature_count": int(len(numeric_features)),
    "categorical_feature_count": int(len(categorical_features)),
    "prepared_feature_count": int(len(feature_names)),
    "training_rows": int(len(X_train)),
    "testing_rows": int(len(X_test)),
    "random_state": RANDOM_STATE,
}

with open(
    OUTPUT_DIR / "preprocessing_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, indent=2)

print("Saved outputs to:")
print(OUTPUT_DIR)


Saved outputs to:
c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\data\processed


## Notebook 02 complete

Completed:

1. Loaded the merged dataset.
2. Removed identifiers and leakage from the model inputs.
3. Kept gender separately for fairness evaluation.
4. Selected early Banner, attendance, and Moodle features.
5. Converted numeric and categorical data types.
6. Created stratified training and testing sets.
7. Imputed missing values.
8. Standardised numeric features.
9. One hot encoded categorical features.
10. Saved the prepared data for baseline modelling.
